# 🤖 MiniMax Face Refine (SAM Jawline Contour) - Autonomous Batch Runner
### Method 2: SAM Semantic Jawline Segmentation
* **Mode:** **Autonomous Headless Batch Runner**
* **Features:** Multi-clip queue, non-blocking fault tolerance, instant output persistence, and auto-termination (`sys.exit(0)`).


In [ ]:
# [CONFIGURATION] Antigravity Batch Video Restoration Jobs
# Modify or inject this job list programmatically before dispatching.

BATCH_CLIPS = [
    {
        "id": "shot_001_wide_recreation",
        "video_input": "H3_SHOT_001.mp4",
        "tracking_ref": "ananya_clip_ref.png",
        "dna_ref": "ananya_dna_front.jpg",
        "denoise": 0.65,
        "steps": 25,
        "prompt": "high quality sharp natural face, detailed facial micro-texture, cinematic realism"
    }
]

print(f"Loaded {len(BATCH_CLIPS)} clip restoration jobs for autonomous processing.", flush=True)


In [ ]:
# [Step 1/4] Setup Environment
import os, sys, subprocess, time

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

if not os.path.exists("ComfyUI"):
    subprocess.run("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git", shell=True, check=True)

custom_nodes = "ComfyUI/custom_nodes"
nodes = [
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "https://github.com/storyicon/comfyui_segment_anything.git",
    "https://github.com/fearnworks/comfyui_fearnworksnodes.git"
]
for n in nodes:
    target = os.path.join(custom_nodes, os.path.splitext(os.path.basename(n))[0])
    if not os.path.exists(target):
        subprocess.run(f"git clone --depth 1 {n} {target}", shell=True, check=True)

pkgs = ["huggingface_hub>=0.28.0", "hf_transfer", "opencv-python-headless", "ultralytics", "imageio", "imageio-ffmpeg"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "ComfyUI/requirements.txt"], check=True)
print("✓ Environment ready.", flush=True)


In [ ]:
# [Step 2/4] Provision Model Weights
import os, shutil, glob
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.makedirs("ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("ComfyUI/models/vae", exist_ok=True)
os.makedirs("ComfyUI/models/sams", exist_ok=True)
os.makedirs("ComfyUI/models/ultralytics/bbox", exist_ok=True)

ds_candidates = glob.glob('/kaggle/input/*minimax*weights*')
ds_root = ds_candidates[0] if ds_candidates else '/kaggle/input/minimax-h3-open-weights'

def link_or_fetch(rel_ds, dst, hf_repo, hf_file):
    src = os.path.join(ds_root, rel_ds)
    if os.path.exists(src):
        if not os.path.exists(dst):
            try: os.symlink(src, dst)
            except Exception: shutil.copyfile(src, dst)
        print(f"✓ [MOUNTED] {dst}", flush=True)
    else:
        if not os.path.exists(dst):
            print(f"⏳ Downloading: {hf_file}...", flush=True)
            p = hf_hub_download(repo_id=hf_repo, filename=hf_file)
            try: os.symlink(p, dst)
            except Exception: shutil.copyfile(p, dst)

link_or_fetch("clip/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "ComfyUI/models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "Comfy-Org/MiniMax-H3", "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors")
link_or_fetch("vae/minimax_h3_video_vae_int8_convrot.safetensors", "ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors", "Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_fp16.safetensors")
link_or_fetch("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "ComfyUI/models/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "Comfy-Org/MiniMax-H3", "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors")

sam_dst = "ComfyUI/models/sams/sam_vit_b.0_1.pth"
link_or_fetch("sams/sam_vit_b.0_1.pth", sam_dst, "fearnworks/vosr2", "sam_vit_b.0_1.pth")
if not os.path.exists(sam_dst):
    subprocess.run(f"wget -q -c https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {sam_dst}", shell=True)

yolo_dst = "ComfyUI/models/ultralytics/bbox/face_yolov8m.pt"
if not os.path.exists(yolo_dst):
    subprocess.run(f"wget -q -c https://github.com/akanametov/yolo-face/releases/download/v0.0.0/face_yolov8m.pt -O {yolo_dst}", shell=True)
print("✓ Weights ready.", flush=True)


In [ ]:
# [Step 3/4] Deploy Pipeline Assets & Launch Headless Server
import subprocess, threading, time, urllib.request, json, shutil

PORT = 8188
OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
input_dir = "ComfyUI/input"
os.makedirs(input_dir, exist_ok=True)

# Deploy workflows and references
local_wf = "workflows"
if os.path.exists(local_wf):
    for f in os.listdir(local_wf):
        p = os.path.join(local_wf, f)
        if f.endswith(".mp4") or f.endswith(".png") or f.endswith(".jpg"):
            shutil.copy(p, input_dir)

comfy_proc = subprocess.Popen(
    f"{sys.executable} ComfyUI/main.py --listen 127.0.0.1 --port {PORT} --enable-cors-header --preview-method none",
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

def monitor():
    for line in comfy_proc.stdout:
        if any(w in line for w in ["Executing", "error", "Error", "CUDA"]):
            print(f"[ComfyUI] {line.strip()}", flush=True)

threading.Thread(target=monitor, daemon=True).start()

server_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/system_stats", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                print("✓ Headless ComfyUI Server online!", flush=True)
                break
    except Exception:
        time.sleep(2)

if not server_ready:
    raise RuntimeError("Server failed to start.")


In [ ]:
# [Step 4/4] Fault-Tolerant Batch Execution & Immediate Auto-Termination
import urllib.request, json, time, shutil

api_wf_file = os.path.join("workflows", "WF2_SAM_Contour_API.json")
with open(api_wf_file, "r", encoding="utf-8") as f:
    template = json.load(f)

results = []
print("=" * 75, flush=True)
print(f">>> Executing Batch Face Refine Queue: {len(BATCH_CLIPS)} Clips", flush=True)
print("=" * 75, flush=True)

for i, job in enumerate(BATCH_CLIPS, 1):
    job_id = job.get("id", f"clip_{i}")
    print(f"\n[{i}/{len(BATCH_CLIPS)}] Restoring: {job_id}...", flush=True)
    t0 = time.time()
    
    try:
        wf = json.loads(json.dumps(template))
        # Parameter overrides
        if "1" in wf and "inputs" in wf["1"]: wf["1"]["inputs"]["video"] = job.get("video_input", "H3_SHOT_001.mp4")
        if "60" in wf and "inputs" in wf["60"]: wf["60"]["inputs"]["image"] = job.get("tracking_ref", "ananya_clip_ref.png")
        if "6" in wf and "inputs" in wf["6"]: wf["6"]["inputs"]["image"] = job.get("dna_ref", "ananya_dna_front.jpg")
        if "7" in wf and "inputs" in wf["7"]: wf["7"]["inputs"]["image"] = job.get("dna_ref", "ananya_dna_front.jpg")
        if "9" in wf and "inputs" in wf["9"]: wf["9"]["inputs"]["prompt"] = job.get("prompt", "")
        if "16" in wf and "inputs" in wf["16"]:
            wf["16"]["inputs"]["denoise"] = float(job.get("denoise", 0.65))
            wf["16"]["inputs"]["steps"] = int(job.get("steps", 25))

        payload = json.dumps({"prompt": wf}).encode("utf-8")
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/prompt", data=payload, headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req) as resp:
            prompt_id = json.loads(resp.read().decode("utf-8"))["prompt_id"]

        done = False
        while not done:
            time.sleep(3)
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/history/{prompt_id}") as h_resp:
                hist = json.loads(h_resp.read().decode("utf-8"))
                if prompt_id in hist:
                    status = hist[prompt_id].get("status", {})
                    if status.get("completed", False):
                        done = True
                    elif status.get("status_str") == "error":
                        raise RuntimeError(f"ComfyUI Error: {status.get('messages')}")

        out_files = []
        comfy_out = "ComfyUI/output"
        if os.path.exists(comfy_out):
            for f in os.listdir(comfy_out):
                if f.endswith(".mp4") or f.endswith(".png"):
                    src_p = os.path.join(comfy_out, f)
                    dst_p = os.path.join(OUTPUT_DIR, f"{job_id}_{f}")
                    shutil.copyfile(src_p, dst_p)
                    out_files.append(dst_p)

        elapsed = time.time() - t0
        print(f"✓ [{job_id}] Restored in {elapsed:.1f}s -> {out_files}", flush=True)
        results.append({"id": job_id, "status": "SUCCESS", "elapsed": f"{elapsed:.1f}s", "path": out_files})

    except Exception as e:
        elapsed = time.time() - t0
        print(f"❌ [{job_id}] Error: {e} (Continuing next clip...)", flush=True)
        results.append({"id": job_id, "status": "FAILED", "elapsed": f"{elapsed:.1f}s", "error": str(e)})

print("\n" + "=" * 80, flush=True)
print("🏁 BATCH RESTORATION FINISHED! SUMMARY:", flush=True)
print("=" * 80, flush=True)
for r in results:
    icon = "✓" if r["status"] == "SUCCESS" else "❌"
    print(f" {icon} {r['id']} ({r['status']} in {r['elapsed']}) -> {r.get('path', r.get('error'))}", flush=True)
print("=" * 80, flush=True)

print("\n>>> Terminating server and exiting to preserve GPU quota. Goodbye!", flush=True)
comfy_proc.terminate()
time.sleep(1)
sys.exit(0)
